In [3]:
# BASELINE MODELS OVERVIEW
import pandas as pd
import numpy as np
import joblib
import os
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, accuracy_score

print("Loading Original Data...")
csv_path = 'pfas_major_compounds.csv'
if not Path(csv_path).exists():
    csv_path = '../data/processed/pfas_major_compounds.csv'

df = pd.read_csv(csv_path)

# Base algorithm: Target = strictly greater or equal to 10.0 on single 'value'
df['is_contaminated'] = (df['value'] >= 10.0).astype(int)

df = df.dropna(subset=['lat', 'lon'])
df = pd.get_dummies(df, columns=['type'], drop_first=True)
features = ['lat', 'lon'] + [c for c in df.columns if 'type_' in c]

model_df = df.dropna(subset=features + ['is_contaminated']).copy()
X = model_df[features]
y = model_df['is_contaminated']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

os.makedirs('base_model', exist_ok=True)

print("\n--- TRAINING BASE MODELS ---")
# Extra Trees
print("1. Extra Trees")
et = ExtraTreesClassifier(n_estimators=100, max_depth=10, class_weight='balanced', random_state=42, n_jobs=-1)
et.fit(X_train, y_train)
joblib.dump(et, 'base_model/base_extratrees.pkl')

# KNN
print("2. KNN")
knn = KNeighborsClassifier(n_neighbors=5, weights='distance', n_jobs=-1)
knn.fit(X_train_scaled, y_train)
joblib.dump(knn, 'base_model/base_knn.pkl')

# Random Forest
print("3. Random Forest")
rf = RandomForestClassifier(n_estimators=100, max_depth=10, class_weight='balanced', random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
joblib.dump(rf, 'base_model/base_randomforest.pkl')

# XGBoost
print("4. XGBoost")
pos_weight = (len(y_train) - y_train.sum()) / max(y_train.sum(), 1)
xgb = XGBClassifier(scale_pos_weight=pos_weight, n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42, eval_metric='logloss')
xgb.fit(X_train, y_train)
joblib.dump(xgb, 'base_model/base_xgboost.pkl')

print("\nALL 4 MODELS SAVED TO 'base_model/' folder.")
print("\n--- BASELINE ALGORITHM RESULTS ---")
results = []
results.append(('Extra Trees', roc_auc_score(y_test, et.predict_proba(X_test)[:, 1]), accuracy_score(y_test, et.predict(X_test))))
results.append(('KNN', roc_auc_score(y_test, knn.predict_proba(X_test_scaled)[:, 1]), accuracy_score(y_test, knn.predict(X_test_scaled))))
results.append(('Random Forest', roc_auc_score(y_test, rf.predict_proba(X_test)[:, 1]), accuracy_score(y_test, rf.predict(X_test))))
results.append(('XGBoost', roc_auc_score(y_test, xgb.predict_proba(X_test)[:, 1]), accuracy_score(y_test, xgb.predict(X_test))))

results.sort(key=lambda x: x[1], reverse=True)
for rank, (name, auc, acc) in enumerate(results):
    print(f"#{rank+1} -> {name:20s} | ROC-AUC: {auc:.4f}  | Accuracy: {acc:.4f}")



Loading Original Data...

--- TRAINING BASE MODELS ---
1. Extra Trees
2. KNN
3. Random Forest
4. XGBoost

ALL 4 MODELS SAVED TO 'base_model/' folder.

--- BASELINE ALGORITHM RESULTS ---
#1 -> KNN                  | ROC-AUC: 0.9229  | Accuracy: 0.8715
#2 -> XGBoost              | ROC-AUC: 0.8978  | Accuracy: 0.8273
#3 -> Random Forest        | ROC-AUC: 0.8959  | Accuracy: 0.8302
#4 -> Extra Trees          | ROC-AUC: 0.7928  | Accuracy: 0.7500
